# Day 4 – Data Standardization

**Project:** Sales & Business Performance Analytics Dashboard  
**Role:** Member 1 – Data Cleaning

## Objective
Standardize date, categorical, text, and relevant fields using Pandas.

The original raw dataset is preserved. All transformations are applied to a separate DataFrame, and the standardized output is saved in `data/processed/`.

## Standardization principles
- Parse dates consistently.
- Remove leading and trailing whitespace from text fields.
- Preserve existing valid category labels and capitalization.
- Preserve identifiers and customer/product names.
- Convert postal codes to string to retain alphanumeric formats and leading zeros.
- Preserve numeric values without unnecessary modifications.


## 1. Import libraries and load the original dataset

In [1]:
import pandas as pd
from pathlib import Path

# Locate the project root from the current working directory.
project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data/raw/sample_-_superstore.xls").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project data/raw folder.")

source_path = project_root / "data/raw/sample_-_superstore.xls"
output_path = project_root / "data/processed/standardized_superstore.csv"

df_original = pd.read_excel(source_path, engine="xlrd")
df = df_original.copy()

print("Project root:", project_root)
print("Original dataset shape:", df_original.shape)
print("Working dataset shape:", df.shape)
print("Original dataset remains unchanged: transformations use a separate copy.")


Project root: /Users/hemanth/Sales-Business-Performance-Analytics-Dashboard
Original dataset shape: (10194, 21)
Working dataset shape: (10194, 21)
Original dataset remains unchanged: transformations use a separate copy.


## 2. Inspect relevant columns before standardization

In [2]:
date_columns = ["Order Date", "Ship Date"]

category_columns = [
    "Ship Mode", "Segment", "Country/Region",
    "Region", "Category", "Sub-Category"
]

postal_column = "Postal Code"

print("Date data types:")
print(df[date_columns].dtypes)

print("\nCategory values before standardization:")
for col in category_columns:
    print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")

print("\nPostal Code data types before standardization:")
print(df[postal_column].map(type).value_counts())


Date data types:
Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

Category values before standardization:
Ship Mode: ['First Class', 'Same Day', 'Second Class', 'Standard Class']
Segment: ['Consumer', 'Corporate', 'Home Office']
Country/Region: ['Canada', 'United States']
Region: ['Central', 'East', 'South', 'West']
Category: ['Furniture', 'Office Supplies', 'Technology']
Sub-Category: ['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']

Postal Code data types before standardization:
Postal Code
<class 'int'>    9983
<class 'str'>     211
Name: count, dtype: int64


## 3. Standardize date fields

In [3]:
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="raise")

print("Date data types after standardization:")
print(df[date_columns].dtypes)

print("\nDate ranges:")
for col in date_columns:
    print(f"{col}: {df[col].min()} to {df[col].max()}")


Date data types after standardization:
Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

Date ranges:
Order Date: 2023-01-03 00:00:00 to 2026-12-30 00:00:00
Ship Date: 2023-01-07 00:00:00 to 2027-01-05 00:00:00


## 4. Standardize text and categorical fields

In [4]:
# Exclude Postal Code because it contains mixed integer and string values.
text_columns = [
    col for col in df.select_dtypes(include=["object", "str", "string"]).columns
    if col != postal_column
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

print("Text columns standardized:", len(text_columns))
print("Postal Code excluded from general text cleaning.")
print("Postal Code non-missing values preserved:", df[postal_column].notna().sum())


Text columns standardized: 13
Postal Code excluded from general text cleaning.
Postal Code non-missing values preserved: 10194


## 5. Standardize postal codes

In [5]:
# Postal codes are identifiers, not numeric measurements.
# String conversion preserves alphanumeric Canadian postal codes
# and leading zeros in values such as 05401.

df[postal_column] = df[postal_column].astype("string").str.strip()

print("Postal Code dtype:", df[postal_column].dtype)
print("Postal Code examples:", df[postal_column].dropna().unique()[:20])


Postal Code dtype: string
Postal Code examples: <StringArray>
['77095', '60540', '19143', '42420', '30605', '90049', '77340', '78041',
 '22153', '19901', '94109', '29464', '43055', '71111', '19140', '30076',
 '19134', '97477', '85254', '72401']
Length: 20, dtype: string


## 6. Validate the standardized dataset

In [6]:
print("DATASET SHAPE")
print("Original:", df_original.shape)
print("Standardized:", df.shape)

print("\nMISSING VALUES")
print(df.isna().sum().to_string())

print("\nDATE VALIDATION")
for col in date_columns:
    print(f"{col} dtype:", df[col].dtype)
    print(f"{col} invalid dates:", pd.to_datetime(df[col], errors="coerce").isna().sum())

print("\nSHIP DATE BEFORE ORDER DATE:",
      (df["Ship Date"] < df["Order Date"]).sum())

print("\nPOSTAL CODE VALIDATION")
print("Postal Code dtype:", df[postal_column].dtype)
print("Missing postal codes:", df[postal_column].isna().sum())

print("\nTEXT WHITESPACE VALIDATION")
for col in text_columns:
    values = df[col].dropna()
    print(f"{col}: {(values != values.str.strip()).sum()} whitespace issues")


DATASET SHAPE
Original: (10194, 21)
Standardized: (10194, 21)

MISSING VALUES
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
Quantity          0
Discount          0
Profit            0

DATE VALIDATION
Order Date dtype: datetime64[us]
Order Date invalid dates: 0
Ship Date dtype: datetime64[us]
Ship Date invalid dates: 0

SHIP DATE BEFORE ORDER DATE: 0

POSTAL CODE VALIDATION
Postal Code dtype: string
Missing postal codes: 0

TEXT WHITESPACE VALIDATION
Order ID: 0 whitespace issues
Ship Mode: 0 whitespace issues
Customer ID: 0 whitespace issues
Customer Name: 0 whitespace issues
Segment: 0 whitespace issues
Country/Region: 0 whitespace issues
City: 0 whitespace issues
State/Provi

## 7. Export the standardized dataset

In [7]:
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)

print("Standardized dataset saved to:", output_path)
print("Output size (bytes):", output_path.stat().st_size)


Standardized dataset saved to: /Users/hemanth/Sales-Business-Performance-Analytics-Dashboard/data/processed/standardized_superstore.csv
Output size (bytes): 2472638


## 8. Final validation and conclusion

In [8]:
# Read the exported CSV and verify its structure.
df_exported = pd.read_csv(
    output_path,
    dtype={"Postal Code": "string"},
    parse_dates=date_columns
)

assert df_exported.shape == df_original.shape
assert list(df_exported.columns) == list(df_original.columns)
assert df_exported["Postal Code"].dtype.name == "string"
assert (df_exported["Ship Date"] >= df_exported["Order Date"]).all()
assert df_exported["Row ID"].equals(df_original["Row ID"])

print("Final validation: PASSED")
print("Original rows:", len(df_original))
print("Exported rows:", len(df_exported))
print("Original columns:", len(df_original.columns))
print("Exported columns:", len(df_exported.columns))
print("Postal Code dtype:", df_exported["Postal Code"].dtype)
print("Output exists:", output_path.exists())


Final validation: PASSED
Original rows: 10194
Exported rows: 10194
Original columns: 21
Exported columns: 21
Postal Code dtype: string
Output exists: True


## Conclusion

The dataset was standardized using Pandas without modifying the original raw source.

Date columns were parsed consistently, text fields were trimmed, existing category labels and identifiers were preserved, and postal codes were converted to string to support both numeric-looking and alphanumeric postal formats.

The standardized CSV is stored separately in `data/processed/`. Validation checks confirm that the exported dataset retains the original row and column structure.
